## 🧠 Challenge: Fine-Tuning VGG16 for Better Performance

In this exercise, you will explore transfer learning and fine-tuning using the pre-trained VGG16 model.

---

### 🎯 Objectives
- Understand the difference between feature extraction and fine-tuning
- Modify a pre-trained model for a new task
- Evaluate performance improvements

---

### 🧩 Tasks

1. **Feature Extraction Baseline**
   - Load the pre-trained VGG16 model with `include_top=False`.
   - Freeze all convolutional layers.
   - Train only the custom classifier (dense layers on top).
   - Record:
     - Training accuracy
     - Validation accuracy

2. **Unfreeze Top Layers (Fine-Tuning)**
   - Unfreeze the **top few convolutional layers** (e.g., last 4–8 layers).
   - Keep earlier layers frozen.
   - Retrain the model with a **low learning rate**.

3. **Compare Results**
   - Compare performance before and after fine-tuning.
   - Did validation accuracy improve?
   - Did overfitting increase or decrease?

---

### 💡 Reflection Questions

- Why do we freeze most layers in transfer learning?
- Why should we use a smaller learning rate during fine-tuning?
- What kind of features do early vs late CNN layers learn?

---

### ⭐ Bonus

- Experiment with unfreezing more or fewer layers.
- Add **data augmentation** and observe its effect.
- Try another pre-trained model (e.g., ResNet50) and compare results.


## Loading the Datasets
* The dataset used in this challenge is the Dogs vs. Cats dataset, which can be obtained from:
    * https://www.microsoft.com/en-us/download/details.aspx?id=54765
* Dataset Preparation Notes:
    * Location: The data should be unzipped and available at ../data/kaggle_dogs_vs_cats/train.
    * Path Adjustments: Note that the exact paths and the names of the image files may differ depending on the source of the dataset. The code below is adjusted to match the current directory structure.

In [1]:
import os, shutil, pathlib

original_dir = pathlib.Path("data/kaggle_dogs_vs_cats/train/PetImages")
new_base_dir = pathlib.Path("data/dogs_vs_cats")

RESET_DATA = False  # True :delete and start over;False: keep existing data if it exists

target_dir = "data/dogs_vs_cats"

if RESET_DATA and os.path.exists(target_dir):
    shutil.rmtree(target_dir)
    print("Old data cleaned.")
elif os.path.exists(target_dir):
    print("Data directory already exists. Skipping deletion.")


def make_subset(subset_name, start_index, end_index):
    for category in ("Cat", "Dog"):
        # create directory
        target_dir = new_base_dir / subset_name / category
        os.makedirs(target_dir, exist_ok=True)
        
        # create file names
        fnames = [f"{i}.jpg" for i in range(start_index, end_index)]
        
        for fname in fnames:
            src = original_dir / category / fname
            dst = target_dir / fname

            if src.exists():
                shutil.copyfile(src, dst)
            else:
                print(f"Warning: {src} not found.")

# split the data into train, validation, and test sets
make_subset("train", start_index=0, end_index=1000)      # train: 0~999
make_subset("validation", start_index=1000, end_index=1500) # validation: 1000~1499
make_subset("test", start_index=1500, end_index=2500)       # test: 1500~2499

print("Done! Data splitting complete.")

Data directory already exists. Skipping deletion.
Done! Data splitting complete.


In [2]:
import os
import pathlib
import tensorflow as tf
image_dataset_from_directory = tf.keras.utils.image_dataset_from_directory

# Set the path to your data folder
train_dir = "data/dogs_vs_cats/train"
val_dir = "data/dogs_vs_cats/validation"
test_dir = "data/dogs_vs_cats/test"

#  training and validation
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(180, 180),
    batch_size=32)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=(180, 180),
    batch_size=32)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(180, 180),
    batch_size=32)

#Force single thread and disable prefetching
options = tf.data.Options()
options.threading.max_intra_op_parallelism = 1
options.threading.private_threadpool_size = 1
options.experimental_distribute.auto_shard_policy = tf.data.experimental.AutoShardPolicy.OFF

train_dataset = train_dataset.with_options(options)
validation_dataset = validation_dataset.with_options(options)
test_dataset = test_dataset.with_options(options)

print("Data set loading complete")


Found 2000 files belonging to 2 classes.
Found 1000 files belonging to 2 classes.
Found 2000 files belonging to 2 classes.
Data set loading complete


## Cleaning the Data

* When the AI tries to read a broken image, it crashes. `InvalidArgumentError Traceback (most recent call last)`
* We use the `PIL` library to check every photo. If a photo is bad, we delete it.

In [3]:
import os
import PIL
from PIL import Image

# Path
base_dir = pathlib.Path("data/dogs_vs_cats")

print("Checking for corrupted images...")
num_skipped = 0

for folder in ("train", "validation", "test"):
    for category in ("Cat", "Dog"):
        folder_path = base_dir / folder / category
        for fname in os.listdir(folder_path):
            fpath = folder_path / fname
            try:
                # Try to open and verify the image
                img = Image.open(fpath)
                img.verify()
            except (IOError, SyntaxError) as e:
                #If the image is broken, delete it
                print(f"Found corrupted file and deleted: {fpath}")
                os.remove(fpath)
                num_skipped += 1

print(f"Checking complete! Deleted {num_skipped} corrupted images.")

Checking for corrupted images...
Found corrupted file and deleted: data\dogs_vs_cats\train\Cat\666.jpg
Checking complete! Deleted 1 corrupted images.


## Feature Extraction Baseline

In [4]:
import os
import shutil
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

train_dir = "data/dogs_vs_cats/train"
val_dir = "data/dogs_vs_cats/validation"

# VGG16 model set
conv_base = keras.applications.vgg16.VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(180, 180, 3))

conv_base.trainable = False

# create the model
inputs = keras.Input(shape=(180, 180, 3))
x = keras.applications.vgg16.preprocess_input(inputs) 
x = conv_base(x)
x = layers.Flatten()(x)
x = layers.Dense(256, activation="relu")(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)

model.compile(loss="binary_crossentropy",
              optimizer="rmsprop",
              metrics=["accuracy"])

# train the model
history = model.fit(
    train_dataset,
    epochs=10,
    validation_data=validation_dataset)

Epoch 1/10
22/63 [=========>....................] - ETA: 26s - loss: 10.9942 - accuracy: 0.8949

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xa8 in position 131: invalid start byte